# Feature Analysis

This notebook will analyze the final feature set selected in the previous notebook and determine any remaining preprocessing considerations before model training.

Planned area:

- **Feature distributions** — analyze feature ranges, scale differences, skewness, and extreme values to determine appropriate preprocessing.

The goal is to make **actionable preprocessing decisions for the training pipeline**, without repeating the feature-selection analysis.

In [ ]:
import numpy as np
import pandas as pd

from src.common.constants import FINAL_SELECTED_FEATURES
from src.modeling.data_utils import load_modeling_data, split_modeling_data

In [ ]:
df = load_modeling_data()

train_df, val_df, test_df = split_modeling_data(df)

print(f"Train:, {len(train_df)}, {train_df["ts"].min()}, →, {train_df["ts"].max()}")

## Experiment 1 — Feature Distribution Analysis

**Goal:** Understand the distributions and scales of the final 23 features to identify potential preprocessing needs for downstream models.

**Potential checks:**
- Feature ranges and scale differences
- Distribution shape and skewness
- Extreme or suspicious values

**Focus:** Determine whether scaling or transformations may be worth considering for Ridge Regression and the Dense NN. No feature selection will be performed here.

In [ ]:
X_train = train_df[FINAL_SELECTED_FEATURES].copy()

distribution_analysis = pd.DataFrame({
    "min": X_train.min(),
    "max": X_train.max(),
    "mean": X_train.mean(),
    "std": X_train.std(),
    "median": X_train.median(),
    "skewness": X_train.skew(),
    "unique": X_train.nunique(),
})

distribution_analysis["range"] = (
    distribution_analysis["max"] - distribution_analysis["min"]
)

distribution_analysis["cv"] = (
    distribution_analysis["std"] / distribution_analysis["mean"].abs()
)

distribution_analysis.round(3)

**Observation:** The final 23 features have substantially different scales, with ranges spanning from `11` (`month`) to over `7,500` (`co_today`). Scaling is therefore necessary for **Ridge Regression** and the **Dense NN**, while **Random Forest** can use the original feature values. Several pollutant and variability features are also strongly right-skewed, but this reflects the natural distribution of the collected air-quality data and does not by itself justify transforming or removing values. The appropriate scaler for the non-tree models still needs to be determined through further analysis.

In [ ]:
print("NaN values:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\nInfinite values:")
print(np.isinf(X_train.select_dtypes(include=np.number)).sum())

**Observation:** The training feature set contains **no missing values** and **no infinite values** across all 23 selected features. Therefore, no missing-value imputation or non-finite value handling is required before model training.

**Observation:** The final 23 features have substantially different scales, so **RobustScaler** was selected for Ridge Regression and the Dense NN because it is less sensitive to the strong skewness and extreme values present in several pollutant features. Random Forest will use the original feature values. Features with very high right-skewness will additionally use **log1p transformation** before scaling to reduce the influence of extreme magnitudes while preserving the underlying observations.